In [ ]:
import gzip
import json
import numpy as np
import matplotlib.pyplot as plt
import re
import seaborn as sns
import os

from matplotlib.backends.backend_pdf import PdfPages

sns.set();
%matplotlib inline

## Preprocess Clustering

 * choose max level of hierarchy
 * choose which sections to include
 * choose whether to ensure uniqueness of IDs

In [ ]:
SPECIAL_SECTIONS = ['INTRODUCTION', 'CONCLUSION', 'PERSPECTIVES']

In [ ]:
def is_box_section(title):
    if '|' not in title:
        return False
    return 'Box' in title.split('|')[0]

In [ ]:
def preprocess_clustering_impl(clustering, max_level, current_level):
    current_level_elements = []
    clusters = []
    num_clusters = 0
    
#     print(max_level, current_level, clustering)
    for el in clustering:
        if isinstance(el, dict):
            inner_clusters, num_inner_clusters = preprocess_clustering_impl(el['references'], 
                                                                            max_level, current_level + 1)
#             print(inner_clusters, num_inner_clusters)
            if max_level <= current_level:
                current_level_elements.extend(inner_clusters)
            else:
                if num_inner_clusters == 1:
                    clusters.append(inner_clusters)
                else:
                    clusters.extend(inner_clusters)
                num_clusters += num_inner_clusters
        else:
            current_level_elements.append(el)
#         print(current_level_elements, clusters)
            
    if max_level <= current_level or not clusters:
        return current_level_elements, 1
    else:
        if current_level_elements:
            clusters.insert(0, current_level_elements)
            num_clusters += 1
        if num_clusters == 1:
            return clusters[0], num_clusters
        else:
            return clusters, num_clusters

def preprocess_clustering(clustering, max_level, 
                          include_special_sections=False,
                          include_box_sections=True):
    flat_clustering = []
    for section in clustering:
        is_special_section = section['title'] in SPECIAL_SECTIONS
        if not include_special_sections and is_special_section:
            continue
        if not include_box_sections and is_box_section(section['title']):
            continue
#         print(section)
        clusters, num_clusters = preprocess_clustering_impl(section['references'], max_level, 1)
#         print(clusters, num_clusters)
        if num_clusters == 1:
            flat_clustering.append(clusters)
        else:
            flat_clustering.extend(clusters)
    return flat_clustering

In [ ]:
def get_clustering_level(clustering):
    levels = []
    for el in clustering:
        if isinstance(el, dict):
            levels.append(get_clustering_level(el['references']))
    if not levels:
        return 1
    else:
        return max(levels) + 1

In [ ]:
def unique_ids_clustering(clustering, method):
    id_cluster = {}
    for i, cluster in enumerate(clustering):
        for pmid in cluster:
            if pmid not in id_cluster:
                id_cluster[pmid] = []
            id_cluster[pmid].append(i)
    if method == 'first_occurrence':
        return {str(k): v[0] for k, v in id_cluster.items()}
    elif method == 'unique_only':
        return {str(k): v[0] for k, v in id_cluster.items() if len(set(v)) == 1}

## Test Clustering Preprocessing

In [ ]:
EXAMPLE_CLUSTERING_1 = [
    {
        'title': 'A',
        'references': [
            134,
            123,
            {
                'title': 'A.1',
                'references': [
                    {
                        'title': 'A.1.1',
                        'references': [
                            123,
                            324,
                        ]
                    },
                    {
                        'title': 'A.1.2',
                        'references': [
                            435
                        ]
                    }
                ]
            }
        ]
    },
    {
        'title': 'B',
        'references': [
            546,
            345
        ]
    },
    {
        'title': 'C',
        'references': [
            {
                'title': 'C.1',
                'references': [
                    234,
                    245
                ]
            }
        ]
    }
]

EXAMPLE_CLUSTERING_2 = [
    {
        'title': 'A',
        'references': [
            14,
            15,
            21
        ]
    },
    {
        'title': 'B',
        'references': [
            33,
            44,
            15
        ]
    }
]

EXAMPLE_CLUSTERING_3 = [
    {
        'title': 'INTRODUCTION',
        'references': [
            1,
            2,
            3
        ]
    },
    {
        'title': 'Box 1 | Box section',
        'references': [
            4,
            5,
            6
        ]
    }
]

In [ ]:
preprocess_clustering(EXAMPLE_CLUSTERING_1, 1)

In [ ]:
preprocess_clustering(EXAMPLE_CLUSTERING_1, 2)

In [ ]:
preprocess_clustering(EXAMPLE_CLUSTERING_1, 3)

In [ ]:
preprocess_clustering(EXAMPLE_CLUSTERING_3, 3)

In [ ]:
preprocess_clustering(EXAMPLE_CLUSTERING_3, 3, include_special_sections=True, include_box_sections=False)

In [ ]:
get_clustering_level(EXAMPLE_CLUSTERING_1)

In [ ]:
get_clustering_level(EXAMPLE_CLUSTERING_2)

In [ ]:
unique_ids_clustering(preprocess_clustering(EXAMPLE_CLUSTERING_1, 1), method='first_occurrence')

In [ ]:
unique_ids_clustering(preprocess_clustering(EXAMPLE_CLUSTERING_1, 1), method='unique_only')

In [ ]:
unique_ids_clustering(preprocess_clustering(EXAMPLE_CLUSTERING_1, 2), method='first_occurrence')

In [ ]:
unique_ids_clustering(preprocess_clustering(EXAMPLE_CLUSTERING_1, 2), method='unique_only')

In [ ]:
unique_ids_clustering(preprocess_clustering(EXAMPLE_CLUSTERING_2, 1), method='first_occurrence')

In [ ]:
unique_ids_clustering(preprocess_clustering(EXAMPLE_CLUSTERING_2, 1), method='unique_only')

## Clustering Overview

In [ ]:
CLUSTERING_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/clustering/'

In [ ]:
with PdfPages('clustering-overview.pdf') as pdf:
    for filename in sorted(os.listdir(CLUSTERING_FOLDER)):
        full_filename = os.path.join(CLUSTERING_FOLDER, filename)
        with open(full_filename, 'r') as f:
            raw_clustering = json.load(f)
        for level in range(1, get_clustering_level(raw_clustering)):
            processed_clustering = preprocess_clustering(raw_clustering, level)
            n_clusters = len(processed_clustering)
            intersections = np.zeros((n_clusters, n_clusters))
            for i in range(n_clusters):
                for j in range(n_clusters):
                    common_elements = set(processed_clustering[i]).intersection(processed_clustering[j])
                    intersections[i, j] = len(common_elements)

            plt.figure(figsize = (10,7))
            sns.heatmap(intersections, annot=True)
            plt.title(f'{filename} - LEVEL {level}')
            pdf.savefig()
            plt.close()

## Estimate PubTrends quality for different clusterings

In [ ]:
from pysrc.papers.analyzer import KeyPaperAnalyzer
from pysrc.papers.pubtrends_config import PubtrendsConfig, AnalyzerSettings
from pysrc.papers.db.loaders import Loaders

from sklearn.metrics.cluster import adjusted_rand_score, v_measure_score, \
                                    adjusted_mutual_info_score, contingency_matrix
from sklearn.metrics import fowlkes_mallows_score

In [ ]:
PUBTRENDS_EXPORT_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/pubtrends_export/'
PUBTRENDS_CONFIG = PubtrendsConfig(test=False)

In [ ]:
def get_clustering(analyzer, target_ids):
    data = analyzer.df[analyzer.df['id'].isin(target_ids)]
    return {k: v for k, v in zip(data['id'], data['comp'])}

In [ ]:
def evaluate_clustering(analyzer, ground_truth, metric):
    actual_clustering = get_clustering(analyzer, ground_truth.keys())
    
    # Align clusterings
    labels_true = []
    labels_pred = []

    for pmid in actual_clustering:
        labels_true.append(ground_truth[pmid])
        labels_pred.append(actual_clustering[pmid])
        
    return metric(labels_true, labels_pred), contingency_matrix(labels_true, labels_pred)

In [ ]:
def reload_exported_analyzer(path_to_archive, source='Pubmed'):
    """
    Load analysis data from json.gz archive generated by PubTrends.
    """
    with gzip.open(path_to_archive, 'rt', encoding='UTF-8') as zipfile:
        data = json.load(zipfile)

    loader, url_prefix = Loaders.get_loader_and_url_prefix(source, PUBTRENDS_CONFIG)
    analyzer = KeyPaperAnalyzer(loader, PUBTRENDS_CONFIG)
    analyzer.init(data)
    
    return analyzer

In [ ]:
def estimate_quality(metric, uniqueness_method, include_box_sections):
    pmids = []
    levels = []
    scores = []
    for filename in sorted(os.listdir(CLUSTERING_FOLDER)):
        pmid, _ = os.path.splitext(filename)
        full_filename = os.path.join(CLUSTERING_FOLDER, filename)
        with open(full_filename, 'r') as f:
            raw_clustering = json.load(f)
        analysis_file = os.path.join(PUBTRENDS_EXPORT_FOLDER, f'{pmid}.json.gz')
        analyzer = reload_exported_analyzer(analysis_file)
        for level in range(1, get_clustering_level(raw_clustering)):
            processed_clustering = preprocess_clustering(raw_clustering, level, 
                                                         include_box_sections=include_box_sections)
            ground_truth = unique_ids_clustering(processed_clustering, method=uniqueness_method)
            score, _ = evaluate_clustering(analyzer, ground_truth, metric)
            print(pmid, level, score)
            pmids.append(pmid)
            levels.append(level)
            scores.append(score)
            
    return pmids, levels, scores 

In [ ]:
COLOR_MAP = {1: '#455cf5', 2: '#33e836', 3: '#eb5021'}

In [ ]:
scores_by_metric = {}

with PdfPages('pubtrends-quality.pdf') as pdf:
    for metric in [adjusted_rand_score, adjusted_mutual_info_score, v_measure_score, fowlkes_mallows_score]:
        for uniqueness_method in ['first_occurrence', 'unique_only']:
            for include_box_sections in [True, False]:
                print(metric.__name__, uniqueness_method, include_box_sections)
                pmids, levels, scores = estimate_quality(metric, uniqueness_method, include_box_sections)
                
                # Visualize
                colors = [COLOR_MAP[l] for l in levels]
    
                plt.figure(figsize=(10, 7))
                plt.scatter(pmids, scores, c=colors)
                plt.title(f'unique: {uniqueness_method} - include_box_sections: {include_box_sections}')
                plt.xlabel('PMID')
                plt.ylabel(metric.__name__)
                plt.xticks(rotation = 90)
                pdf.savefig()
                plt.close()
                
                if metric.__name__ not in scores_by_metric:
                    scores_by_metric[metric.__name__] = scores
                else:
                    scores_by_metric[metric.__name__].append(scores)

In [ ]:
import pandas as pd
from pandas.plotting import scatter_matrix

In [ ]:
scores_df = pd.DataFrame(scores_by_metric).apply(pd.to_numeric, errors='coerce')

In [ ]:
scatter_matrix(scores_df, alpha=0.8, grid=True, figsize=(15, 15))

In [ ]:
scores_df.head()

### Visualize Contingency Matrices for Default PubTrends Settings

In [ ]:
METRIC_ALIAS = {
    'adjusted_rand_score': 'Rand',
    'adjusted_mutual_info_score': 'Mutual-Info',
    'v_measure_score': 'V-Measure',
    'fowlkes_mallows_score': 'Fowlkes-Mallows'
}

with PdfPages('pubtrends-contingency-matrices.pdf') as pdf:
    for filename in sorted(os.listdir(CLUSTERING_FOLDER)):
        pmid, _ = os.path.splitext(filename)
        full_filename = os.path.join(CLUSTERING_FOLDER, filename)
        with open(full_filename, 'r') as f:
            raw_clustering = json.load(f)
        analysis_file = os.path.join(PUBTRENDS_EXPORT_FOLDER, f'{pmid}.json.gz')
        analyzer = reload_exported_analyzer(analysis_file)
        for level in range(1, get_clustering_level(raw_clustering)):
            processed_clustering = preprocess_clustering(raw_clustering, level, 
                                                         include_box_sections=False)
            ground_truth = unique_ids_clustering(processed_clustering, method='unique_only')
            
            scores = {}
            for metric in [adjusted_rand_score, adjusted_mutual_info_score, 
                           v_measure_score, fowlkes_mallows_score]:
                scores[metric.__name__], cm = evaluate_clustering(analyzer, ground_truth, metric)
            
            metrics_str = ", ".join([f'{METRIC_ALIAS[k]}: {v:.2f}' for k, v in scores.items()])
            plt.figure(figsize = (10,7))
            sns.heatmap(cm, annot=True)
            plt.title(f'{filename} - LEVEL {level}\n{metrics_str}')
            plt.xlabel('PubTrends')
            plt.ylabel('Nature Reviews')
            pdf.savefig()
            plt.close()

### Topic Analysis Only for Subgraph (Direct Reference Papers, One of the Subtopics, etc)

In [ ]:
def get_direct_references_subgraph(analyzer, pmid):
    references = list(analyzer.citations_graph.successors(pmid))
    references.append(pmid)
    
    references_similarity_graph = analyzer.similarity_graph.subgraph(references)
    return references_similarity_graph

In [ ]:
from collections import Counter

def get_most_references_subtopic_subgraph(analyzer, pmid):
    references = list(analyzer.citations_graph.successors(pmid))
    reference_comps = analyzer.df[analyzer.df['id'].isin(references)]['comp'].values
    subtopic_id = Counter(reference_comps).most_common(1)[0][0]
    subtopic_papers = analyzer.df[analyzer.df['comp'] == subtopic_id]['id'].values
    
    subtopic_subgraph = analyzer.similarity_graph.subgraph(subtopic_papers)
    return subtopic_subgraph

In [ ]:
def evaluate_clustering_for_partition(partition, ground_truth, pmid, metric):
    # Get clustering
    actual_clustering = {k: v for k, v in partition.items() if k in ground_truth}
    
    # Align clusterings
    labels_true = []
    labels_pred = []

    for pmid in actual_clustering:
        labels_true.append(ground_truth[pmid])
        labels_pred.append(actual_clustering[pmid])
        
    return metric(labels_true, labels_pred), contingency_matrix(labels_true, labels_pred)

In [ ]:
def recalculate_topic_analysis(analyzer, graph=None):
    if not graph:
        graph = analyzer.similarity_graph
    settings = AnalyzerSettings()
    topics_dendrogram, partition, comp_other, components, comp_sizes = \
                analyzer.topic_analysis(graph,
                                    topic_min_size=settings.TOPIC_MIN_SIZE,
                                    max_topics_number=settings.TOPICS_MAX_NUMBER,
                                    random_state=settings.SEED,
                                    similarity_bibliographic_coupling=settings.SIMILARITY_BIBLIOGRAPHIC_COUPLING,
                                    similarity_cocitation=settings.SIMILARITY_COCITATION,
                                    similarity_citation=settings.SIMILARITY_CITATION,
                                    similarity_text_citation=settings.SIMILARITY_TEXT_CITATION)
    return partition

#### Direct References

In [ ]:
METRIC_ALIAS = {
    'adjusted_rand_score': 'Rand',
    'adjusted_mutual_info_score': 'Mutual-Info',
    'v_measure_score': 'V-Measure',
    'fowlkes_mallows_score': 'Fowlkes-Mallows'
}

with PdfPages('pubtrends-contingency-matrices-refs-only.pdf') as pdf:
    for filename in sorted(os.listdir(CLUSTERING_FOLDER)):
        pmid, _ = os.path.splitext(filename)
        full_filename = os.path.join(CLUSTERING_FOLDER, filename)
        with open(full_filename, 'r') as f:
            raw_clustering = json.load(f)
        analysis_file = os.path.join(PUBTRENDS_EXPORT_FOLDER, f'{pmid}.json.gz')
        analyzer = reload_exported_analyzer(analysis_file)
        direct_references_subgraph = get_direct_references_subgraph(analyzer, pmid)
        partition = recalculate_topic_analysis(analyzer, direct_references_subgraph)
        for level in range(1, get_clustering_level(raw_clustering)):
            processed_clustering = preprocess_clustering(raw_clustering, level, 
                                                         include_box_sections=False)
            ground_truth = unique_ids_clustering(processed_clustering, method='unique_only')
            
            scores = {}
            for metric in [adjusted_rand_score, adjusted_mutual_info_score, 
                           v_measure_score, fowlkes_mallows_score]:
                scores[metric.__name__], cm = evaluate_clustering_for_partition(partition, 
                                                                                ground_truth, 
                                                                                pmid, metric)
            
            metrics_str = ", ".join([f'{METRIC_ALIAS[k]}: {v:.2f}' for k, v in scores.items()])
            plt.figure(figsize = (10,7))
            sns.heatmap(cm, annot=True)
            plt.title(f'{filename} - LEVEL {level}\n{metrics_str}')
            plt.xlabel('PubTrends')
            plt.ylabel('Nature Reviews')
            pdf.savefig()
            plt.close()

#### The Biggest Subtopic

In [ ]:
METRIC_ALIAS = {
    'adjusted_rand_score': 'Rand',
    'adjusted_mutual_info_score': 'Mutual-Info',
    'v_measure_score': 'V-Measure',
    'fowlkes_mallows_score': 'Fowlkes-Mallows'
}

with PdfPages('pubtrends-contingency-matrices-biggest-subtopic.pdf') as pdf:
    for filename in sorted(os.listdir(CLUSTERING_FOLDER)):
        pmid, _ = os.path.splitext(filename)
        full_filename = os.path.join(CLUSTERING_FOLDER, filename)
        with open(full_filename, 'r') as f:
            raw_clustering = json.load(f)
        analysis_file = os.path.join(PUBTRENDS_EXPORT_FOLDER, f'{pmid}.json.gz')
        analyzer = reload_exported_analyzer(analysis_file)
        subtopic_subgraph = get_most_references_subtopic_subgraph(analyzer, pmid)
        if not subtopic_subgraph:
            print('Bad subgraph')
        partition = recalculate_topic_analysis(analyzer, subtopic_subgraph)
        for level in range(1, get_clustering_level(raw_clustering)):
            processed_clustering = preprocess_clustering(raw_clustering, level, 
                                                         include_box_sections=False)
            ground_truth = unique_ids_clustering(processed_clustering, method='unique_only')
            
            scores = {}
            for metric in [adjusted_rand_score, adjusted_mutual_info_score, 
                           v_measure_score, fowlkes_mallows_score]:
                scores[metric.__name__], cm = evaluate_clustering_for_partition(partition, 
                                                                                ground_truth, 
                                                                                pmid, metric)
            
            metrics_str = ", ".join([f'{METRIC_ALIAS[k]}: {v:.2f}' for k, v in scores.items()])
            plt.figure(figsize = (10,7))
            sns.heatmap(cm, annot=True)
            plt.title(f'{filename} - LEVEL {level}\n{metrics_str}')
            plt.xlabel('PubTrends')
            plt.ylabel('Nature Reviews')
            pdf.savefig()
            plt.close()

### PubTrends Cluster Sizes

TODO: compare with only-refs setting

In [ ]:
from collections import Counter

for filename in sorted(os.listdir(CLUSTERING_FOLDER)):
    pmid, _ = os.path.splitext(filename)
    full_filename = os.path.join(CLUSTERING_FOLDER, filename)
    with open(full_filename, 'r') as f:
        raw_clustering = json.load(f)
    analysis_file = os.path.join(PUBTRENDS_EXPORT_FOLDER, f'{pmid}.json.gz')
    analyzer = reload_exported_analyzer(analysis_file)
    processed_clustering = preprocess_clustering(raw_clustering, level, 
                                                 include_box_sections=False)
    ground_truth = unique_ids_clustering(processed_clustering, method='unique_only')
    actual_clustering = get_clustering(analyzer, ground_truth.keys()).values()
    comp_sizes = Counter(actual_clustering).most_common(3)
    n_papers = len(actual_clustering)
    scaled_sizes = [v * 100 / n_papers for comp, v in comp_sizes]
    print(f"{pmid}\t{scaled_sizes}")

## PubTrends Export Overview

In [ ]:
import pandas as pd

In [ ]:
with PdfPages('pubtrends-similarity-parameters-hist.pdf') as pdf:
    for filename in sorted(os.listdir(PUBTRENDS_EXPORT_FOLDER)):
        analysis_file = os.path.join(PUBTRENDS_EXPORT_FOLDER, filename)
        analyzer = reload_exported_analyzer(analysis_file)
        stats = {
            'cocitation': [],
            'bibcoupling': [],
            'citation': [],
            'text': []
        }
        for _, _, data in analyzer.similarity_graph.edges(data=True):
            for key in stats.keys():
                stats[key].append(data.get(key, 0))

        stats_df = pd.DataFrame(stats).apply(pd.to_numeric, errors='coerce')
        ax = stats_df.hist(log=True)
        plt.suptitle(filename)
        pdf.savefig()
        plt.close()